# AI Lecture Intelligence — Full Stack on Google Colab

This notebook runs both the FastAPI backend and React frontend on Google Colab.

## What this does:
1. Installs Node.js and npm for React frontend
2. Clones/uploads your project
3. Installs Python dependencies (FastAPI backend)
4. Installs Node dependencies (React frontend)
5. Starts both servers with public URLs via ngrok

## Requirements:
- Google Colab account (free)
- ngrok account (free) for public URLs: https://ngrok.com

## Step 1: Install Node.js and npm

In [ ]:
%%bash
# Install Node.js 20.x LTS
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs

# Verify installation
node --version
npm --version

## Step 2: Clone or Upload Project

**Option A:** Clone from GitHub (if your repo is public):

In [ ]:
# Option A: Clone from GitHub
!rm -rf /content/AI-Lecture-Intelligence /content/speech_rag
!git clone https://github.com/rishiwalia08/AI-Lecture-Intelligence.git
%cd /content/AI-Lecture-Intelligence

# Verify frontend files exist
!echo "Current dir:" && pwd
!ls -la
!ls -la frontend || true
!test -f frontend/package.json && echo "✅ frontend/package.json found" || echo "❌ frontend/package.json missing in repo"

**Option B:** Upload project as ZIP file:
1. Compress your local `speech_rag` folder to `speech_rag.zip`
2. Run the cell below and upload the ZIP file

In [ ]:
# Option B: Upload ZIP
from google.colab import files
uploaded = files.upload()  # Upload speech_rag.zip

!unzip -q speech_rag.zip
%cd speech_rag

## Step 3: Install Python Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 4: Set Environment Variables

In [ ]:
import os

# Hugging Face API Key (get from https://huggingface.co/settings/tokens)
os.environ['HF_API_KEY'] = 'hf_YOUR_TOKEN_HERE'  # ⚠️ REPLACE THIS

# Optional: Groq API Key for faster inference
# os.environ['GROQ_API_KEY'] = 'gsk_YOUR_TOKEN_HERE'

print("✅ Environment variables set")

## Step 5: Install Frontend Dependencies

In [ ]:
%%bash
set -e

if [ -f /content/AI-Lecture-Intelligence/frontend/package.json ]; then
  cd /content/AI-Lecture-Intelligence/frontend
elif [ -f /content/speech_rag/frontend/package.json ]; then
  cd /content/speech_rag/frontend
else
  echo "❌ frontend/package.json not found."
  echo "Push frontend files to GitHub, then re-run Step 2."
  exit 1
fi

pwd
npm install

## Step 6: Install ngrok (for public URLs)

In [ ]:
# Install pyngrok
!pip install -q pyngrok

from pyngrok import ngrok, conf

# Set your ngrok auth token (get free token from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ⚠️ REPLACE THIS
conf.get_default().auth_token = NGROK_AUTH_TOKEN

print("✅ ngrok configured")

## Step 7: Start Backend (FastAPI)

This runs the backend server in the background.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Start FastAPI backend in background
backend_process = subprocess.Popen(
    ["uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Starting FastAPI backend...")
time.sleep(10)  # Wait for backend to initialize

# Create ngrok tunnel for backend
backend_tunnel = ngrok.connect(8000, bind_tls=True)
backend_url = backend_tunnel.public_url

print(f"\n✅ Backend running at: {backend_url}")
print(f"   API docs: {backend_url}/docs")

## Step 8: Configure Frontend to Use Backend URL

In [ ]:
# Create .env file for frontend with backend URL
with open('frontend/.env', 'w') as f:
    f.write(f'VITE_API_BASE_URL={backend_url}\n')

print(f"✅ Frontend configured to use backend: {backend_url}")

## Step 9: Start Frontend (React + Vite)

This runs the React development server.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Start Vite dev server in background
frontend_process = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    cwd="frontend",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Starting React frontend...")
time.sleep(8)  # Wait for Vite to start

# Create ngrok tunnel for frontend
frontend_tunnel = ngrok.connect(5173, bind_tls=True)
frontend_url = frontend_tunnel.public_url

print(f"\n✅ Frontend running at: {frontend_url}")
print(f"\n🎉 Open this URL in your browser: {frontend_url}")

## Step 10: Monitor Services

Keep this cell running to maintain the servers.

In [ ]:
import time

print("\n" + "="*60)
print("🚀 SERVICES RUNNING")
print("="*60)
print(f"\n📱 Frontend UI:  {frontend_url}")
print(f"🔧 Backend API:  {backend_url}")
print(f"📚 API Docs:     {backend_url}/docs")
print("\n⚠️  Keep this cell running to maintain the servers.")
print("⚠️  Stop this cell (⏹) to shut down all services.\n")
print("="*60)

try:
    while True:
        time.sleep(60)
        print("✅ Services still running...", flush=True)
except KeyboardInterrupt:
    print("\n🛑 Shutting down services...")
    backend_process.terminate()
    frontend_process.terminate()
    ngrok.kill()
    print("✅ All services stopped.")

## Optional: Check Service Status

In [ ]:
import requests

# Test backend health
try:
    response = requests.get(f"{backend_url}/health", timeout=5)
    print("✅ Backend Status:", response.json())
except Exception as e:
    print("❌ Backend Error:", e)

# Test frontend
try:
    response = requests.get(frontend_url, timeout=5)
    print(f"✅ Frontend Status: {response.status_code}")
except Exception as e:
    print("❌ Frontend Error:", e)

## Troubleshooting

### Backend not starting?
- Check your HF_API_KEY is set correctly
- Ensure all dependencies installed: `!pip list | grep fastapi`
- Check logs: `!tail -20 /tmp/backend.log`

### Frontend not loading?
- Verify Node.js installed: `!node --version`
- Check npm packages: `!cd frontend && npm list react`
- Ensure .env file created: `!cat frontend/.env`

### ngrok issues?
- Verify auth token is correct
- Check active tunnels: `!curl http://localhost:4040/api/tunnels`
- Try restarting ngrok: `ngrok.kill()` then restart cells

### Out of memory?
- Use Colab Pro for more RAM
- Reduce model size in config.yaml
- Close other Colab notebooks

## Cleanup (Run when done)

In [ ]:
# Stop all services
try:
    backend_process.terminate()
    frontend_process.terminate()
    ngrok.kill()
    print("✅ All services stopped")
except:
    print("⚠️  Services may already be stopped")